# HLS ingest

Select STAC records from the MAAP HLS archive, virtualize every band and
resolution level of the COGs they describe, and commit a batch of granules,
metadata rows and virtual arrays together in one Icechunk commit.

Needs a `~/.netrc` entry for `urs.earthdata.nasa.gov` and AWS credentials that
can read `s3://nasa-maap-data-store`.

**Set `ACCESS` below.** `"s3"` is the fast path but LP DAAC's protected bucket
is readable that way only from AWS `us-west-2`; `"https"` works from anywhere.
The choice affects only how COG headers are fetched here -- references are
written relative to a named container (`vcc://lpdaac/...`), so this store is
readable either way whichever mode built it.


In [2]:
# 1. Open the archive and look at what a record holds.
import logging

from icechest.demo.source import open_archive, select_granules

# "s3" is faster but readable only from AWS us-west-2; "https" works anywhere.
# It decides how bytes are fetched here, not what the store records.
ACCESS = "https"

# The archive's parquet dictionary-encodes some columns; Iceberg has no
# dictionary type, so PyIceberg reads them as strings and says so, loudly.
logging.getLogger("pyiceberg.io.pyarrow").setLevel(logging.ERROR)

archive = open_archive()
rows = select_granules(archive, limit=5)
rows.select(["id", "datetime", "proj:epsg", "proj:shape", "proj:transform"]).to_pandas()


,id,datetime,proj:epsg,proj:shape,proj:transform
0,HLS.L30.T20JKP.2026004T142004.v2.0,2026-01-04 14:20:04.576000+00:00,32620,"[3660, 3660]","[30.0, 0.0, 199980.0, 0.0, -30.0, -3099960.0, ..."
1,HLS.L30.T44QLM.2026018T051400.v2.0,2026-01-18 05:14:00.270000+00:00,32644,"[3660, 3660]","[30.0, 0.0, 300000.0, 0.0, -30.0, 2700000.0, 0..."
2,HLS.L30.T44QLM.2026011T050747.v2.0,2026-01-11 05:07:47.856000+00:00,32644,"[3660, 3660]","[30.0, 0.0, 300000.0, 0.0, -30.0, 2700000.0, 0..."
3,HLS.L30.T32TML.2026017T101215.v2.0,2026-01-17 10:12:15.482000+00:00,32632,"[3660, 3660]","[30.0, 0.0, 399960.0, 0.0, -30.0, 4600020.0, 0..."
4,HLS.L30.T13SFA.2026027T172600.v2.0,2026-01-27 17:26:00.012000+00:00,32613,"[3660, 3660]","[30.0, 0.0, 600000.0, 0.0, -30.0, 4100040.0, 0..."


In [7]:
# 2. Two URLs per asset: one to read through now, one to record.
from icechest.demo.assets import asset_urls, to_vcc_url

row = rows.to_pylist()[0]
read_url = asset_urls(row, access=ACCESS)["B04"]

# The reference carries no endpoint, so a reader resolves it against
# whichever container their own store declares.
print("read through:", read_url)
print("recorded as: ", to_vcc_url(row["assets"]["B04"]["href"]))


{'B01': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B01.tif',
 'B02': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B02.tif',
 'B03': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B03.tif',
 'B04': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B04.tif',
 'B05': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B05.tif',
 'B06': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B06.tif',
 'B07': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B07.tif',
 'B09': 's3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B09.tif',
 'B10': 's3://lp

In [8]:
# 3. How many resolution levels this COG carries.
from icechest.demo.credentials import object_store_registry
from icechest.demo.tiff import parse_ifds
from icechest.demo.virtualize import read_header

registry = object_store_registry(ACCESS)
parse_ifds(read_header(asset_urls(row, access=ACCESS)["B04"], registry))


ValueError: Could not find an ObjectStore matching the url `s3://lp-prod-protected/HLSL30.020/HLS.L30.T20JKP.2026004T142004.v2.0/HLS.L30.T20JKP.2026004T142004.v2.0.B04.tif`

In [ ]:
# 4. The convention attributes, from the record alone.
from icechest.demo.conventions import granule_attrs

granule_attrs(
    epsg=row["proj:epsg"],
    shape=row["proj:shape"],
    transform=row["proj:transform"],
    levels=5,
)

In [ ]:
# 5. Open a local store and declare the table from the archive's own schema.
from pathlib import Path
from icechest.demo.store import ensure_table, open_store

repo = open_store(Path("hls-demo-store"), access=ACCESS)
ensure_table(repo, archive.schema())
repo.read("main").pointers

In [ ]:
# 6. Ingest the batch. One commit carries the rows and every virtual array.
from icechest.demo.ingest import ingest_batch

result = ingest_batch(repo, rows, registry=registry, access=ACCESS)
result.snapshot_id, len(result.committed), result.skipped

In [ ]:
# 7. Query the metadata, then follow a row to its pixels.
snap = repo.read("main")
table = snap.table("granules").scan().to_arrow()
print(table.select(["id", "stac_hash", "stac_hash_block", "array_path"]).to_pandas())

granule = result.committed[0]
snap.group[f"{granule}/B04/multiscales/0"][:16, :16]